In [1]:
print("Hi")

Hi


In [ ]:
import requests
import pandas as pd
import time


API_KEY = "Your_Api_Key"

LISTING_API_URL = "https://api.hasdata.com/scrape/airbnb/listing"
PROPERTY_API_URL = "https://api.hasdata.com/scrape/airbnb/property"

headers = {
    "x-api-key": API_KEY
}

cities = [
    "Chennai",
    "Bangalore",
    "Hyderabad",
    "Mumbai",
    "New Delhi"
]

CHECK_IN = "2026-07-01"
CHECK_OUT = "2026-07-05"
ADULTS = 1

MAX_PER_CITY = 100
SLEEP_BETWEEN_REQUESTS = 1

all_properties = []
seen_ids = set()


def safe_get(data, *keys):
    current = data

    for key in keys:
        if not isinstance(current, dict):
            return None

        current = current.get(key)

        if current is None:
            return None

    return current


def get_property_details(url):
    if not url:
        return {}

    try:
        response = requests.get(
            PROPERTY_API_URL,
            headers=headers,
            params={"url": url},
            timeout=60
        )
    except requests.exceptions.RequestException as e:
        print(f"Property request failed: {e}")
        return {}

    if response.status_code != 200:
        print(f"Property API error: {response.status_code}")
        print(response.text[:300])
        return {}

    try:
        return response.json()
    except ValueError:
        print("Could not parse property response as JSON")
        return {}


def extract_amenities(details):
    amenities = details.get("amenities")

    if not amenities:
        return ""

    if isinstance(amenities, list):
        cleaned = []

        for item in amenities:
            if isinstance(item, dict):
                name = item.get("title") or item.get("name")
                if name:
                    cleaned.append(name)
            elif isinstance(item, str):
                cleaned.append(item)

        return ",".join(cleaned)

    return ""


def normalize_property_details(details):
    host = details.get("host") or {}

    return {
        "bedrooms": details.get("bedrooms"),
        "bathrooms": details.get("bathrooms"),
        "beds": details.get("beds"),
        "guest_capacity": (
            details.get("personCapacity")
            or details.get("guestCapacity")
            or details.get("guests")
        ),
        "property_type": (
            details.get("propertyType")
            or details.get("propertyTypeName")
        ),
        "room_type": details.get("roomType"),
        "is_superhost": (
            host.get("isSuperhost")
            or details.get("isSuperhost")
        ),
        "host_name": host.get("name"),
        "host_id": host.get("id"),
        "host_rating": host.get("rating"),
        "host_reviews": host.get("reviewsCount"),
        "amenities": extract_amenities(details),
        "description": details.get("description")
    }


for city in cities:
    print(f"\nCollecting {city} listings...")

    city_count = 0
    next_page_token = None
    no_new_pages = 0
    page_number = 1

    while city_count < MAX_PER_CITY:
        params = {
            "location": city,
            "checkIn": CHECK_IN,
            "checkOut": CHECK_OUT,
            "adults": ADULTS
        }

        if next_page_token:
            params["nextPageToken"] = next_page_token

        try:
            response = requests.get(
                LISTING_API_URL,
                headers=headers,
                params=params,
                timeout=60
            )
        except requests.exceptions.RequestException as e:
            print(f"Listing request failed for {city}: {e}")
            break

        if response.status_code != 200:
            print(f"Listing API error for {city}: {response.status_code}")
            print(response.text[:500])
            break

        data = response.json()
        properties = data.get("properties", [])

        if not properties:
            print(f"No more properties found for {city}")
            break

        before_city_count = city_count

        for prop in properties:
            if city_count >= MAX_PER_CITY:
                break

            listing_id = prop.get("id")

            if not listing_id or listing_id in seen_ids:
                continue

            seen_ids.add(listing_id)

            price = prop.get("price") or {}
            listing_url = prop.get("url")

            print(
                f"Fetching details for {city} listing "
                f"{city_count + 1}/{MAX_PER_CITY}"
            )

            details = get_property_details(listing_url)
            normalized_details = normalize_property_details(details)

            row = {
                "city": city,
                "id": listing_id,
                "title": prop.get("title"),
                "latitude": prop.get("latitude"),
                "longitude": prop.get("longitude"),
                "rating": prop.get("rating"),
                "reviews": prop.get("reviews"),
                "badges": ",".join(prop.get("badges") or []),
                "price_original": price.get("originalPrice"),
                "price_discounted": price.get("discountedPrice"),
                "url": listing_url
            }

            row.update(normalized_details)

            all_properties.append(row)
            city_count += 1

            time.sleep(SLEEP_BETWEEN_REQUESTS)

        new_count = city_count - before_city_count

        print(
            f"{city} | Page {page_number} | "
            f"New: {new_count} | City Total: {city_count} | "
            f"Overall Total: {len(all_properties)}"
        )

        if city_count >= MAX_PER_CITY:
            print(f"Reached {MAX_PER_CITY} listings for {city}")
            break

        if new_count == 0:
            no_new_pages += 1
        else:
            no_new_pages = 0

        if no_new_pages >= 3:
            print(f"Stopping {city}: 3 pages with no new listings")
            break

        next_page_token = data.get("pagination", {}).get("nextPageToken")

        if not next_page_token:
            print(f"No next page for {city}")
            break

        page_number += 1
        time.sleep(SLEEP_BETWEEN_REQUESTS)


df = pd.DataFrame(all_properties)

df.to_csv("airbnb_india_500_enriched_listings.csv", index=False)

print("\nFinished")
print("Total Listings:", len(df))
print("Saved as airbnb_india_500_enriched_listings.csv")

print("\nListings per city:")
print(df["city"].value_counts())

print("\nColumns:")
print(df.columns.tolist())


Fetching details for Chennai listing 1/100
Fetching details for Chennai listing 2/100
Fetching details for Chennai listing 3/100
Fetching details for Chennai listing 4/100
Fetching details for Chennai listing 5/100
Fetching details for Chennai listing 6/100
Fetching details for Chennai listing 7/100
Fetching details for Chennai listing 8/100
Fetching details for Chennai listing 9/100
Fetching details for Chennai listing 10/100
Fetching details for Chennai listing 11/100
Fetching details for Chennai listing 12/100
Fetching details for Chennai listing 13/100
Fetching details for Chennai listing 14/100
Fetching details for Chennai listing 15/100
Fetching details for Chennai listing 16/100
Fetching details for Chennai listing 17/100
Fetching details for Chennai listing 18/100
Chennai | Page 1 | New: 18 | City Total: 18 | Overall Total: 18
Fetching details for Chennai listing 19/100
Fetching details for Chennai listing 20/100
Fetching details for Chennai listing 21/100
Fetching details for

In [1]:
import json
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
df = pd.read_csv("airbnb_india_500_enriched_listings_final.csv")
df.head(3)

,city,id,title,latitude,longitude,rating,reviews,badges,price_original,price_discounted,...,guest_capacity,property_type,room_type,is_superhost,host_name,host_id,host_rating,host_reviews,amenities,description
0,Chennai,1281837406334000727,Room in West Mambalam,13.054200,80.22490,4.94,119.0,Guest favorite,$79,NaN,...,NaN,NaN,NaN,True,Architha,NaN,4.94,NaN,"Paid washer – In building, Iron, Air condition...","A family friendly, spacious bedroom with an at..."
1,Chennai,1099604073674013447,Home in Thiruvanmiyur,12.985700,80.25760,4.98,58.0,Guest favorite,$165,NaN,...,3.0,NaN,NaN,True,Prashanth,NaN,4.92,NaN,"Hair dryer, Washer, Essentials, Hangers, Bed l...",Spacious 1bhk House in an independent compound...
2,Chennai,1513726421969655644,Home in Chennai,12.982908,80.18936,4.92,13.0,Guest favorite,$138,NaN,...,4.0,NaN,NaN,NaN,Ram,NaN,4.97,NaN,"Hair dryer, Shampoo, Body soap, Hot water, Sho...",What makes my place special is the positive en...


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   city              500 non-null    str    
 1   id                500 non-null    int64  
 2   title             500 non-null    str    
 3   latitude          500 non-null    float64
 4   longitude         500 non-null    float64
 5   rating            465 non-null    float64
 6   reviews           465 non-null    float64
 7   badges            430 non-null    str    
 8   price_original    500 non-null    str    
 9   price_discounted  77 non-null     str    
 10  url               500 non-null    str    
 11  bedrooms          446 non-null    float64
 12  bathrooms         416 non-null    float64
 13  beds              500 non-null    int64  
 14  guest_capacity    442 non-null    float64
 15  property_type     0 non-null      float64
 16  room_type         0 non-null      float64
 17  is_super

In [5]:
df.isnull().sum()

city                  0
id                    0
title                 0
latitude              0
longitude             0
rating               35
reviews              35
badges               70
price_original        0
price_discounted    423
url                   0
bedrooms             54
bathrooms            84
beds                  0
guest_capacity       58
property_type       500
room_type           500
is_superhost        118
host_name             0
host_id             500
host_rating           5
host_reviews        500
amenities             0
description           0
dtype: int64

In [6]:
def clean_price(value):
    if pd.isna(value):
        return np.nan
    cleaned = re.sub(r'[^0-9.]', '', str(value))
    return float(cleaned) if cleaned else np.nan

def count_amenities(value):
    if pd.isna(value) or str(value).strip() == '':
        return 0
    return len([item for item in str(value).split(',') if item.strip()])

def has_text(value, keyword):
    if pd.isna(value):
        return 0
    return int(keyword.lower() in str(value).lower())

def infer_property_type(title):
    if pd.isna(title):
        return 'Unknown'
    text = str(title).lower()
    if 'room' in text:
        return 'Room'
    if 'apartment' in text or 'flat' in text:
        return 'Apartment'
    if 'villa' in text:
        return 'Villa'
    if 'home' in text or 'house' in text:
        return 'House'
    if 'condo' in text:
        return 'Condo'
    return 'Other'

In [7]:
df['price'] = df['price_original'].apply(clean_price)
df = df.dropna(subset=['price']).copy()
upper_limit = df['price'].quantile(0.99)
df = df[(df['price'] > 0) & (df['price'] <= upper_limit)].copy()

df['amenities_count'] = df['amenities'].apply(count_amenities)
df['has_wifi'] = df['amenities'].apply(lambda x: has_text(x, 'wifi'))
df['has_kitchen'] = df['amenities'].apply(lambda x: has_text(x, 'kitchen'))
df['has_ac'] = df['amenities'].apply(lambda x: has_text(x, 'air conditioning'))
df['has_parking'] = df['amenities'].apply(lambda x: has_text(x, 'parking'))
df['has_washer'] = df['amenities'].apply(lambda x: has_text(x, 'washer'))
df['has_tv'] = df['amenities'].apply(lambda x: has_text(x, 'tv'))
df['has_pool'] = df['amenities'].apply(lambda x: has_text(x, 'pool'))
df['has_workspace'] = df['amenities'].apply(lambda x: has_text(x, 'workspace'))
df['description_length'] = df['description'].fillna('').astype(str).str.len()
df['title_length'] = df['title'].fillna('').astype(str).str.len()
df['property_type_inferred'] = df['title'].apply(infer_property_type)
df['is_superhost'] = df['is_superhost'].fillna(False).astype(bool).astype(int)

In [8]:
df.shape

(495, 37)

In [9]:
display(df[['city', 'price', 'bedrooms', 'bathrooms', 'beds', 'guest_capacity', 'amenities_count', 'property_type_inferred']].head())

,city,price,bedrooms,bathrooms,beds,guest_capacity,amenities_count,property_type_inferred
0,Chennai,79.0,NaN,NaN,1,NaN,24,Room
1,Chennai,165.0,1.0,1.0,1,3.0,26,House
2,Chennai,138.0,2.0,2.0,2,4.0,37,House
3,Chennai,205.0,2.0,2.0,2,4.0,28,Apartment
4,Chennai,207.0,2.0,2.0,2,6.0,43,Apartment


In [10]:
feature_columns = [
    'city', 'latitude', 'longitude', 'rating', 'reviews', 'bedrooms', 'bathrooms',
    'beds', 'guest_capacity', 'is_superhost', 'host_rating', 'amenities_count',
    'has_wifi', 'has_kitchen', 'has_ac', 'has_parking', 'has_washer', 'has_tv',
    'has_pool', 'has_workspace', 'description_length', 'title_length', 'property_type_inferred'
]

X = df[feature_columns]
y = df['price']



In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=X['city'])

print('Train rows:', X_train.shape[0])
print('Test rows:', X_test.shape[0])

Train rows: 396
Test rows: 99


In [12]:
numeric_features = [
    'latitude', 'longitude', 'rating', 'reviews', 'bedrooms', 'bathrooms', 'beds',
    'guest_capacity', 'is_superhost', 'host_rating', 'amenities_count', 'has_wifi',
    'has_kitchen', 'has_ac', 'has_parking', 'has_washer', 'has_tv', 'has_pool',
    'has_workspace', 'description_length', 'title_length'
]
categorical_features = ['city', 'property_type_inferred']


In [13]:
def build_preprocessor():
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])
    return ColumnTransformer([
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ])

In [15]:
models = {
    'Ridge Regression': TransformedTargetRegressor(
        regressor=Pipeline([
            ('preprocessor', build_preprocessor()),
            ('model', Ridge(alpha=50.0))
        ]),
        func=np.log1p,
        inverse_func=np.expm1
    ),
    'Random Forest': TransformedTargetRegressor(
        regressor=Pipeline([
            ('preprocessor', build_preprocessor()),
            ('model', RandomForestRegressor(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            ))
        ]),
        func=np.log1p,
        inverse_func=np.expm1
    )
}

In [16]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    return {
        'train_r2': r2_score(y_train, train_pred),
        'test_r2': r2_score(y_test, test_pred),
        'train_mae': mean_absolute_error(y_train, train_pred),
        'test_mae': mean_absolute_error(y_test, test_pred),
        'train_rmse': mean_squared_error(y_train, train_pred) ** 0.5,
        'test_rmse': mean_squared_error(y_test, test_pred) ** 0.5,
        'overfit_gap_r2': r2_score(y_train, train_pred) - r2_score(y_test, test_pred)
    }

In [17]:
metrics = {}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    result = evaluate_model(model, X_train, X_test, y_train, y_test)
    cv_mae = -cross_val_score(model, X_train, y_train, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
    result['cv_mae_mean'] = cv_mae.mean()
    result['cv_mae_std'] = cv_mae.std()
    result['fit_status'] = ('Likely overfitting' if result['overfit_gap_r2'] > 0.25 else 'Likely underfitting' if result['test_r2'] < 0.15 and result['train_r2'] < 0.25 else 'Acceptable')
    metrics[name] = result

results_df = pd.DataFrame(metrics).T.sort_values('test_mae')
display(results_df)

,train_r2,test_r2,train_mae,test_mae,train_rmse,test_rmse,overfit_gap_r2,cv_mae_mean,cv_mae_std,fit_status
Ridge Regression,0.533108,0.349369,59.775085,54.459273,82.334774,75.552277,0.183738,64.859944,6.691295,Acceptable
Random Forest,0.766987,0.300528,40.980885,55.839969,58.165457,78.336711,0.466458,61.105866,3.629687,Likely overfitting


In [18]:
best_model_name = min(metrics, key=lambda name: metrics[name]['test_mae'])
best_model = models[best_model_name]
best_model.fit(X_train, y_train)
test_predictions = best_model.predict(X_test)

print('Best model:', best_model_name)
print('Test MAE:', round(mean_absolute_error(y_test, test_predictions), 2))
print('Test RMSE:', round(mean_squared_error(y_test, test_predictions) ** 0.5, 2))
print('Test R2:', round(r2_score(y_test, test_predictions), 3))

Best model: Ridge Regression
Test MAE: 54.46
Test RMSE: 75.55
Test R2: 0.349


In [20]:
monitoring_rows = []
for column in X_train.columns:
    train_missing = X_train[column].isna().mean() * 100
    test_missing = X_test[column].isna().mean() * 100
    monitoring_rows.append({
        'check': 'missing_percent',
        'feature': column,
        'train_value': round(train_missing, 2),
        'current_value': round(test_missing, 2),
        'status': 'OK' if test_missing <= max(20, train_missing + 10) else 'WARNING'
    })

monitoring_rows.append({
    'check': 'prediction_mae',
    'feature': 'target_price',
    'train_value': None,
    'current_value': round(mean_absolute_error(y_test, test_predictions), 2),
    'status': 'OK' if mean_absolute_error(y_test, test_predictions) < df['price'].median() * 0.5 else 'WARNING'
})

monitoring_rows.append({
    'check': 'prediction_range',
    'feature': 'predicted_price',
    'train_value': f"{df['price'].min():.2f} to {df['price'].max():.2f}",
    'current_value': f"{test_predictions.min():.2f} to {test_predictions.max():.2f}",
    'status': 'OK' if test_predictions.min() > 0 and test_predictions.max() < df['price'].max() * 1.5 else 'WARNING'
})

monitoring_report = pd.DataFrame(monitoring_rows)
monitoring_report.to_csv("airbnb_monitoring_report.csv", index=False)
display(monitoring_report.head(15))

,check,feature,train_value,current_value,status
0,missing_percent,city,0.0,0.0,OK
1,missing_percent,latitude,0.0,0.0,OK
2,missing_percent,longitude,0.0,0.0,OK
3,missing_percent,rating,7.32,5.05,OK
4,missing_percent,reviews,7.32,5.05,OK
5,missing_percent,bedrooms,9.85,15.15,OK
6,missing_percent,bathrooms,16.41,17.17,OK
7,missing_percent,beds,0.0,0.0,OK
8,missing_percent,guest_capacity,11.36,12.12,OK
9,missing_percent,is_superhost,0.0,0.0,OK


In [21]:
artifact = {
    'model': best_model,
    'model_name': best_model_name,
    'feature_columns': feature_columns,
    'target': 'price',
    'metrics': metrics,
}

joblib.dump(artifact, "airbnb_price_prediction_model.joblib")

with open("airbnb_training_metrics.json", 'w') as f:
    json.dump({
        'best_model': best_model_name,
        'rows_used': int(len(df)),
        'train_rows': int(len(X_train)),
        'test_rows': int(len(X_test)),
        'metrics': metrics,
        'model_path': str("airbnb_price_prediction_model.joblib"),
        'monitoring_path': str("airbnb_monitoring_report.csv")
    }, f, indent=2)

print('Saved model:', "airbnb_price_prediction_model.joblib")

Saved model: airbnb_price_prediction_model.joblib


In [22]:
loaded_artifact = joblib.load("airbnb_price_prediction_model.joblib")
loaded_model = loaded_artifact['model']
sample_input = X_test.iloc[[0]]
prediction = loaded_model.predict(sample_input)[0]

print('Prediction test successful')
print('Predicted price:', round(prediction, 2))
print('Actual price:', float(y_test.iloc[0]))
display(sample_input)

Prediction test successful
Predicted price: 186.53
Actual price: 135.0


,city,latitude,longitude,rating,reviews,bedrooms,bathrooms,beds,guest_capacity,is_superhost,...,has_kitchen,has_ac,has_parking,has_washer,has_tv,has_pool,has_workspace,description_length,title_length,property_type_inferred
390,Mumbai,19.05109,72.82582,4.6,126.0,NaN,NaN,1,NaN,0,...,1,1,1,1,1,0,1,2273,25,Room
